In [1]:
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, save_and_log_checkpoint, load_training_checkpoint
import mlflow
import torch
import json
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKTransformerAdapter
from model_classes import WaitKTransformerMT, SimulTransformerConfig

/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_transformer_distillation")

from transformers import AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [4]:
model_cfg = SimulTransformerConfig(
    vocab_size=len(tokenizer),
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048,
    dropout=0.1,
    max_seq_len=64,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

In [8]:
transformer = WaitKTransformerMT(model_cfg)
count_parameters(transformer)

Total parameters:     306,558,976
Trainable parameters: 306,558,976
Frozen parameters:    0


{'total': 306558976, 'trainable': 306558976, 'frozen': 0}

In [13]:
train_cfg = TrainConfig(
    epochs=8,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=1e-4,
    use_amp=True,
)

In [11]:
student, optim, scaler, model_cfg, train_cfg, state, train_time = load_training_checkpoint("./models/transformer_wait10.pt", SimulTransformerConfig, WaitKTransformerMT)
save_and_log_checkpoint(
    path="./checkpoints/transformer_epoch_4.pt",
    student=student,
    optimizer=optim,
    scaler=scaler,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    **state,
    train_time=train_time,
    mlflow_run_id="da30c73b168e4bf296c9597de3623de8"
)

In [6]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [14]:
train_waitk_student(
    student=transformer,
    train_dataset=dataset,
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="transformer",
    resume_from_checkpoint="./checkpoints/transformer_epoch_4.pt"
)

epoch 6/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 7/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 8/8:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/1/runs/da30c73b168e4bf296c9597de3623de8
🧪 View experiment at: http://localhost:5000/#/experiments/1


OptimizedModule(
  (_orig_mod): WaitKTransformerMT(
    (src_embedding): Embedding(256204, 512, padding_idx=1)
    (tgt_embedding): Embedding(256204, 512, padding_idx=1)
    (src_pos): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embedding): Embedding(64, 512)
    )
    (tgt_pos): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embedding): Embedding(64, 512)
    )
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): Lay

In [3]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

In [4]:
transformer = load_training_checkpoint("./models/transformer.pt", SimulTransformerConfig, WaitKTransformerMT)[0]

In [5]:
adapter = WaitKTransformerAdapter(
    model=transformer,
    tokenizer=tokenizer,
    name="transformer_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(
        use_comet=True,
        comet_model_name="Unbabel/wmt22-comet-da",
        comet_batch_size=512,
    ),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63,
        dataset_fraction=0.2
    )
    
    print(result.metrics)
    
    with open(f"./metrics/transformer_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/root/LinearSimultMT/.venv-mamba/lib/python3.12/site-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Validating transformer_wait_k, wait_k=3:   0%|          | 0/61 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/

{'BLEU': 28.03211591918374, 'chrF++': 51.533982239489504, 'TER': 64.83201740931244, 'COMET': 0.7705645415633228, 'AP': 0.6599331161299984, 'AL': 5.1354822455818585, 'DAL': 5.9194275749997125, 'LAAL': 5.456245626186718, 'ATD_text': 3.7085361880103966, 'total_time_sec': 194.41029191800044, 'ms_per_sentence': 3.141629099221105, 'target_tokens_per_sec': 10691.707622540453, 'source_tokens_per_sec': 8951.218491734975, 'first_token_latency_sec': 3.0665777820090376, 'peak_gpu_memory_mb': 10784.95361328125, 'dataset_fraction': 0.2, 'eval_dataset_size': 61882, 'full_dataset_size': 309410, 'generation_total_time_sec': 186.1567630050049, 'generation_ms_per_sentence': 3.008253821870736, 'generation_target_tokens_per_sec': 11165.739919661777}


...

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 121/121 [05:08<00:00, 

{'BLEU': 29.873770656073944, 'chrF++': 53.07475750026356, 'TER': 61.023134649158074, 'COMET': 0.7983951335979057, 'AP': 0.8089576976839544, 'AL': 10.562355470906882, 'DAL': 11.424417765846163, 'LAAL': 9.39735035881629, 'ATD_text': 7.379250903509719, 'total_time_sec': 208.71737950899842, 'ms_per_sentence': 3.372828601354165, 'target_tokens_per_sec': 9704.740471373505, 'source_tokens_per_sec': 8337.633426089342, 'first_token_latency_sec': 3.2903471375006106, 'peak_gpu_memory_mb': 10472.93017578125, 'dataset_fraction': 0.2, 'eval_dataset_size': 61882, 'full_dataset_size': 309410, 'generation_total_time_sec': 199.7331819320243, 'generation_ms_per_sentence': 3.227645873307655, 'generation_target_tokens_per_sec': 10141.269369499954}


Validating transformer_wait_k, wait_k=11:   0%|          | 0/61 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 121/121 [05:08<00:00, 

{'BLEU': 29.889475783855506, 'chrF++': 53.06821204702473, 'TER': 60.902214584468815, 'COMET': 0.7989660362107699, 'AP': 0.8462996549441085, 'AL': 12.325539510623138, 'DAL': 13.17494855938315, 'LAAL': 10.446266665577111, 'ATD_text': 8.401643112063635, 'total_time_sec': 213.44665724600054, 'ms_per_sentence': 3.4492527268995916, 'target_tokens_per_sec': 9469.611874363862, 'source_tokens_per_sec': 8152.898820028756, 'first_token_latency_sec': 3.3565940081064096, 'peak_gpu_memory_mb': 10472.93017578125, 'dataset_fraction': 0.2, 'eval_dataset_size': 61882, 'full_dataset_size': 309410, 'generation_total_time_sec': 203.75223734198516, 'generation_ms_per_sentence': 3.2925929566268892, 'generation_target_tokens_per_sec': 9920.170822995424}
